<style>
.rendered_html, .jp-RenderedHTMLCommon {font-family: "Segoe UI", Arial, sans-serif;}
.rendered_html p, .jp-RenderedHTMLCommon p {text-align: justify; line-height: 1.65;}
.rendered_html h1, .jp-RenderedHTMLCommon h1 {color: #0f3d56; border-bottom: 3px solid #22a06b; padding-bottom: .25em;}
.rendered_html h2, .jp-RenderedHTMLCommon h2 {color: #155e75; border-bottom: 1px solid #cbd5e1; padding-bottom: .2em;}
.rendered_html h3, .jp-RenderedHTMLCommon h3 {color: #166534;}
.rendered_html table, .jp-RenderedHTMLCommon table {font-size: 92%;}
.rendered_html th, .jp-RenderedHTMLCommon th {background: #e6f4f1;}
.callout {border-left: 5px solid #0f766e; background: #ecfdf5; padding: 12px 16px; border-radius: 6px; margin: 12px 0;}
.warning {border-left: 5px solid #dc2626; background: #fff1f2; padding: 12px 16px; border-radius: 6px; margin: 12px 0;}
.note {border-left: 5px solid #2563eb; background: #eff6ff; padding: 12px 16px; border-radius: 6px; margin: 12px 0;}
.hero {background: linear-gradient(135deg,#083344,#0f766e); color:white; padding:32px; border-radius:14px; margin-bottom:18px;}
.hero h1 {color:white; border:0; margin:0 0 8px 0;}
.hero p {text-align:left; margin:4px 0;}
code {color:#7c2d12;}
.diagram-caption, .table-caption {text-align:center !important; color:#475569; font-size:92%; margin:.55em 0 1.25em;}
</style>

<div class="hero">
  <h1>Smart Factory Applied Simulation System Documentation</h1>
  <p><b>Scope:</b> Three.js simulation, WebXR augmented reality, MQTT integration, ESP8266 control and Node-RED dashboards</p>
  <p><b>Document date:</b> 31 July 2026</p>
</div>

This documentation explains what each subsystem does, how information moves through the system, how the augmented-reality mode works, and how an operator uses and tests the solution.


## Documentation note

<div class="note"><b>Credential handling:</b> Network names, passwords and broker credentials are intentionally not reproduced in this notebook. They must be stored and distributed separately.</div>

The production HTML is optimized and deliberately contains no HTML, CSS or JavaScript comments. This notebook is therefore the readable technical explanation and maintenance reference for that compact source file.


## Public repository release note

The repository copy of the simulation has MQTT disabled by default. The original browser HTML contained broker connection credentials; these were removed from index.html. For private integration, copy config.example.js to config.local.js and set a dedicated broker hostname and restricted credentials. The local configuration is excluded from Git. Browser-delivered JavaScript remains visible to every page visitor, so do not deploy privileged or long-lived credentials in it. Rotate the credentials present in the original HTML before reusing the broker.

Desktop simulation and supported WebXR interactions can run without MQTT. Sensor telemetry and physical ESP8266 control require a configured broker and the companion hardware/Node-RED setup.


## Contents

1. [Assessment objective and implementation criteria](#1.-Assessment-objective-and-implementation-criteria)  
2. [System purpose and scope](#2.-System-purpose-and-scope)  
3. [System architecture](#3.-System-architecture)  
4. [Technology stack](#4.-Technology-stack)  
5. [Three-dimensional factory implementation](#5.-Three-dimensional-factory-implementation)  
6. [Functional subsystems](#6.-Functional-subsystems)  
7. [Desktop mode](#7.-Desktop-mode)  
8. [Augmented-reality implementation](#8.-Augmented-reality-implementation)  
9. [MQTT and physical-system integration](#9.-MQTT-and-physical-system-integration)  
10. [ESP8266 hardware mapping](#10.-ESP8266-hardware-mapping)  
11. [Node-RED dashboards](#11.-Node-RED-dashboards)  
12. [How to use the system](#12.-How-to-use-the-system)  
13. [Deployment and production requirements](#13.-Deployment-and-production-requirements)  
14. [Testing and verification](#14.-Testing-and-verification)  
15. [Troubleshooting](#15.-Troubleshooting)  
16. [Maintenance reference](#16.-Maintenance-reference)  
17. [References and official documentation](#17.-References-and-official-documentation)  
- [Appendix A: AI-supported review prompts](#Appendix-A.-AI-supported-review-prompts)  
- [Appendix B: HTML source-code organization and explanation](#Appendix-B.-HTML-source-code-organization-and-explanation)

### List of figures

- **Figure 3.1:** Smart Factory architecture and communication relationships.
- **Figure 8.1:** WebXR augmented-reality lifecycle.
- **Figure 8.2:** Three-times AR movement with collision-safe substeps.

### List of tables

- **Table 3.1:** Architectural layers.
- **Table 4.1:** Technology stack.
- **Table 7.1:** Desktop navigation controls.
- **Table 8.1:** Augmented-reality controls.
- **Table 9.1:** MQTT topic map.
- **Table 10.1:** ESP8266/NodeMCU hardware mapping.
- **Table 10.2:** Controller timing intervals.
- **Table 14.1:** Desktop acceptance tests.
- **Table 14.2:** Augmented-reality acceptance tests.
- **Table 15.1:** Troubleshooting guide.
- **Table 16.1:** Important production constants.
- **Table A.1:** AI-supported review prompts and purposes.
- **Table B.1:** Functional organization of the production HTML source.


# 1. Assessment objective and implementation criteria

The Smart Factory addresses the rubric through a working system rather than a collection of isolated demonstrations. Geometry and imported models form the factory; animation and interaction drive the robot, conveyor and door; WebXR supplies AR; MQTT connects software to real hardware; and responsive browser delivery supports multiple devices.



# 2. System purpose and scope

The project is a browser-based smart-factory digital environment with two connected rooms. Room 1 contains a pick-and-place robot cell and an automated conveyor packaging line. Room 2 contains operator workstations, status displays, chairs, racks and stored materials. An automatic internal sliding door connects the spaces.

The same simulated machines can be controlled in three ways:

- by clicking the 3D control panels in desktop mode;
- by using the four on-screen controls during an AR session;
- by pressing physical buttons connected to an ESP8266/NodeMCU.

MQTT synchronizes the browser, physical controller and Node-RED dashboards. Sensor measurements from a DHT11 and LDR are published to the broker and shown in the software interfaces. The digital scene also contains animation, sound, collision protection, automatic doors, status indicators, imported GLB assets and an optional wireframe view.

## 2.1 Main objectives

- Provide an interactive digital representation of a smart factory.
- Demonstrate robot and conveyor behavior with realistic visual and audio feedback.
- Connect browser simulation and physical hardware through MQTT.
- Visualize actuator states and sensor values in Node-RED.
- Allow an AR-compatible phone to anchor the factory to a selected real horizontal surface.
- Preserve a consistent starting position between desktop and AR modes.
- Prevent navigation through virtual walls, machines and obstacles.
- Operate on ordinary browsers while enabling AR only on supported devices.


# 3. System architecture


```mermaid
flowchart TB
    subgraph DIGITAL["Digital and monitoring layer"]
        direction LR
        B["Browser Smart Factory\nThree.js • Desktop • WebXR AR"] <-->|"WSS publish / subscribe"| H["HiveMQ Cloud\nSecure MQTT broker"]
        H -->|"Live telemetry"| N["Node-RED\nStatus and actuator dashboards"]
    end

    H <-->|"TLS MQTT"| E["ESP8266 / NodeMCU\nMQTT and control logic"]
    E <-->|"GPIO / ADC"| P["Physical layer\nButtons • LEDs • motor • DHT11 • LDR"]

    classDef browser fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px,color:#111827;
    classDef broker fill:#fef3c7,stroke:#b45309,stroke-width:2px,color:#111827;
    classDef dashboard fill:#fce7f3,stroke:#be185d,stroke-width:2px,color:#111827;
    classDef edge fill:#dcfce7,stroke:#15803d,stroke-width:2px,color:#111827;
    classDef physical fill:#ede9fe,stroke:#6d28d9,stroke-width:2px,color:#111827;
    class B browser;
    class H broker;
    class N dashboard;
    class E edge;
    class P physical;
```

<p class="diagram-caption"><b>Figure 3.1:</b> Smart Factory architecture and communication relationships.</p>


## 3.1 Architectural layers

<p class="table-caption"><b>Table 3.1:</b> Architectural layers.</p>

| Layer | Main responsibility | Key implementation |
|---|---|---|
| Presentation | Render the 3D factory and user controls | HTML, CSS, Three.js, WebGL |
| XR | Detect a real surface and place the factory | WebXR `immersive-ar`, hit testing, DOM overlay |
| Simulation | Update robot, conveyor, door, boxes, lights and sound | One animation loop with time-based state updates |
| Messaging | Distribute commands, states and sensor values | HiveMQ Cloud using MQTT |
| Edge control | Read buttons/sensors and drive motor/LED outputs | ESP8266/NodeMCU firmware |
| Monitoring | Aggregate and display live state | Node-RED dashboard flows |
| Physical process | Represent actual machine inputs and outputs | L298N motor driver, buttons, LEDs, DHT11 and LDR |

## 3.2 Bidirectional control

A command from the browser is published to the MQTT broker and received by the ESP8266. The ESP8266 changes its hardware outputs and publishes the resulting state. The browser and Node-RED dashboards subscribe to those state topics, so the digital and physical representations remain aligned. A physical button press follows the reverse path: the ESP8266 publishes the input event, after which the browser animation and dashboards update.


# 4. Technology stack

<p class="table-caption"><b>Table 4.1:</b> Technology stack.</p>

| Technology | Use in the project |
|---|---|
| HTML5 and CSS | Page structure, desktop instructions, AR overlays and responsive controls |
| JavaScript modules | Application logic and event handling |
| Three.js 0.160.0 | Scene graph, camera, geometry, materials, lights, animation and WebGL rendering |
| GLTFLoader | Loading the three embedded GLB model assets |
| WebXR Device API | Immersive AR session creation and XR camera tracking |
| WebXR Hit Test | Horizontal surface detection and placement reticle |
| MQTT.js | Browser connection to HiveMQ over secure WebSocket |
| HiveMQ Cloud | Central MQTT message broker |
| ESP8266 / NodeMCU | Edge controller for buttons, LEDs, motor and sensors |
| PubSubClient | MQTT client in the Arduino firmware |
| WiFiClientSecure | TLS network transport on the ESP8266 |
| Node-RED | Dashboard aggregation and live visualization |
| GLB / glTF | Portable external 3D model format |
| MP3 | Robot and conveyor machine sounds |

The HTML file is largely self-contained because GLB and MP3 assets are embedded as data URIs. Three.js, `GLTFLoader` and MQTT.js are loaded from public content-delivery networks, so an internet connection is still required unless those libraries are hosted locally.


# 5. Three-dimensional factory implementation

## 5.1 Scene graph

The application creates a `THREE.Scene`, a perspective camera and a WebGL renderer. A parent group named **AR factory placement root** holds the complete factory group. Desktop mode leaves this root at the world origin. AR placement moves and rotates the root so the selected real surface becomes the factory reference.

The perspective camera uses:

- field of view: `75°`;
- near clipping plane: `0.05`;
- far clipping plane: `140`;
- desktop position: `(8, 1.7, 25)`.

Rendering uses antialiasing, transparent-background capability, a pixel-ratio cap of `2`, soft PCF shadows and an XR framebuffer scale factor of `0.85` to balance visual quality and mobile performance.

## 5.2 Geometry and models

Procedural geometry uses box, cylinder, plane, ring and torus primitives. These are combined into floors, concrete walls, roof trusses, lights, pipes, fences, control cabinets, conveyor structures, tables, racks, pallets, safety elements and many other factory components. Three embedded GLB assets add externally created models, including the robot assembly, workstation setup and office chair.

Reusable helper constructors create repeated boxes, cylinders, text panels, control panels, conveyor rollers, workstations and collision entries. Cloning is used where the same external model appears multiple times.

## 5.3 Appearance and realism

`MeshStandardMaterial` and `MeshPhysicalMaterial` provide physically based appearance. Generated canvas textures produce concrete, metal and hazard patterns. Roughness, metalness, transparency and transmission differentiate concrete, steel, glass, plastic and painted surfaces.

Realism is strengthened by:

- hemisphere, directional, point and strip lighting;
- soft shadows;
- scene fog in desktop mode;
- animated belt texture and rotating rollers;
- robot and conveyor audio;
- distance-based sound reduction between rooms;
- floor joints, wear, hazard markings and safety stripes;
- animated indicator lights and physical-looking control panels.

The **Show Meshes** button applies a scene-wide wireframe override. It is useful for demonstrating geometry construction and inspecting mesh density without altering the normal shaded scene.


# 6. Functional subsystems

## 6.1 Pick-and-place robot

The robot executes a synchronized pick-and-place cycle. Starting the robot resets the cycle timer, starts the hydraulic sound, changes the control-panel status, displays the carried carton and animates the base, joints and claws. The cycle lasts approximately `7.34 s`, matching the embedded sound. Stopping the robot pauses the sound, clears the running state and returns the mechanism to its initial pose.

## 6.2 Conveyor packaging line

Starting the conveyor performs four coordinated actions:

1. the looping conveyor sound starts;
2. the rollers rotate;
3. the belt texture scrolls;
4. a package appears and moves along the line.

When the package reaches the end, it disappears and a new package is generated after the feed interval. Stopping the conveyor pauses all movement while keeping the package at its current position.

## 6.3 Automatic internal door

The sliding door opens when the user approaches within its open distance and closes only after the user moves beyond the close distance. Before closing, the program checks that the collision body is clear of the doorway. The collider is recalculated because the door itself moves.

## 6.4 Sensors and status

Temperature, humidity, light intensity and motor state arrive over MQTT. Status displays and control-panel LEDs change only when the corresponding state changes, reducing unnecessary work. Invisible sensor meshes support scene logic without appearing in the final visual presentation.

## 6.5 Collision system

The user is represented by an axis-aligned collision body extending approximately `0.32` scene units from the camera in the X and Z directions and from `0.05` to `1.85` vertically. Static and dynamic scene objects contribute collision boxes. X and Z movement are resolved independently, which allows the user to slide along a wall instead of stopping completely.


# 7. Desktop mode

Desktop mode is the default operating state and is available on ordinary modern browsers.

## 7.1 Navigation controls

<p class="table-caption"><b>Table 7.1:</b> Desktop navigation controls.</p>

| Action | Control |
|---|---|
| Lock the mouse | Click inside the 3D scene |
| Look around | Move the mouse |
| Move forward | `W` or `↑` |
| Move backward | `S` or `↓` |
| Move left | `A` or `←` |
| Move right | `D` or `→` |
| Move faster | Hold either `Shift` key |
| Release pointer lock | `Esc` |

Normal movement speed is `3.2` scene units per second and Shift movement is `5.2` units per second. Frame delta is capped at `0.04 s`, preventing a delayed browser frame from producing an excessively large movement jump.

## 7.2 Operating machines

Aim the center crosshair at the green or red control-panel button and click. The green button starts the associated machine, while the red button stops it. The command changes the local simulation immediately and publishes an MQTT command when the broker connection is active.

## 7.3 Desktop collision behavior

Each frame builds an intended movement vector from the keyboard state. The X movement is tested first, followed by the Z movement. A colliding component is rejected while a non-colliding component is retained. This is why the camera moves naturally along walls and machines but cannot pass through them.


# 8. Augmented-reality implementation


```mermaid
flowchart TD
    A["1. Desktop mode\nInitial camera and desktop controls"] --> B["2. AR support gate\nHTTPS, WebXR and immersive-ar"]
    B --> C["3. Start session\nRear camera and hit test"]
    C --> D["4. Detect floor\nHorizontal-plane reticle"]
    D --> E["5. Select surface\nLock floor height and camera heading"]
    E --> F["6. Run in AR\nFour controls, Exit AR and 3× travel"]
    F --> G["7. End session\nRestore the desktop start pose"]

    classDef start fill:#e0f2fe,stroke:#0369a1,stroke-width:2px,color:#111827;
    classDef process fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px,color:#111827;
    classDef active fill:#dcfce7,stroke:#15803d,stroke-width:2px,color:#111827;
    classDef finish fill:#ede9fe,stroke:#6d28d9,stroke-width:2px,color:#111827;
    class A start;
    class B,C,D,E process;
    class F active;
    class G finish;
```

<p class="diagram-caption"><b>Figure 8.1:</b> WebXR augmented-reality lifecycle.</p>


## 8.1 What the AR mode does

AR mode uses the phone's rear camera and WebXR tracking. The user selects a real horizontal surface, and that surface becomes the height and orientation reference for the virtual factory. The virtual factory floor, safety stripes and hazard markings remain fixed to the selected plane. The implementation does **not** continuously rescan or replace the real floor texture after placement; it locks a plane anchor and stops hit testing.

The phone begins inside the virtual factory at the same logical location used on desktop. This avoids placing the entire building several metres ahead and then forcing the user to walk into it.

## 8.2 Compatibility gate

Before enabling **Start AR**, the program verifies:

1. the page is running in a secure context (`HTTPS`);
2. `navigator.xr` exists;
3. `navigator.xr.isSessionSupported("immersive-ar")` returns `true`.

Unsupported devices retain the normal desktop instructions. A small screen or touch display alone is not treated as proof of AR support.

Typical supported configuration:

- ARCore-compatible Android phone;
- current Google Chrome;
- current Google Play Services for AR;
- camera permission enabled;
- application served from HTTPS.

## 8.3 Session creation

Pressing **Start AR** requests:

```javascript
navigator.xr.requestSession("immersive-ar", {
  requiredFeatures: ["hit-test"],
  optionalFeatures: ["dom-overlay", "local-floor"],
  domOverlay: { root: document.body }
})
```

`hit-test` is mandatory because the factory must be placed on a detected surface. `dom-overlay` allows the four machine controls and **Exit AR** to remain visible over the camera view. `local-floor` is optional because device support varies.

During session startup, the program:

- leaves pointer lock;
- clears held desktop keys;
- resets AR collision history;
- shows the AR overlay and Exit button;
- removes the desktop background and fog;
- disables expensive shadow rendering;
- hides selected roof and outer-wall meshes to improve visibility from inside;
- hides the factory until placement is complete.


## 8.4 Surface detection and selection

A viewer reference space is requested and used to create a hit-test source. Every XR frame returns possible intersections between the camera ray and the real environment. A green ring reticle appears only when a sufficiently horizontal plane is detected; the plane-orientation check requires the Y component of the hit-test matrix to be greater than approximately `0.85`.

Recommended scanning procedure:

1. point the phone downward at a well-lit floor;
2. move the phone slowly so ARCore can recognize visual features;
3. wait for the green circle;
4. face the direction that should correspond to the desktop starting view;
5. tap the green circle once.

After the tap, hit testing is cancelled and the reticle is hidden. The selected surface is therefore stable and is no longer changed by later camera movement.

## 8.5 Placement mathematics

Let:

- $\mathbf{p}_{xr}$ be the current XR camera position;
- $y_f$ be the selected floor height;
- $\theta$ be the horizontal camera heading;
- $R_y(\theta)$ be a rotation around the vertical axis;
- $\mathbf{p}_{desktop}=(8,0,25)$ be the desktop starting point projected onto the floor.

The factory root rotation is:

$$
R_{factory}=R_y(\theta)
$$

and its translation is:

$$
\mathbf{t}_{factory}=
\begin{bmatrix}p_{xr,x}\\y_f\\p_{xr,z}\end{bmatrix}
-R_y(\theta)\mathbf{p}_{desktop}.
$$

Therefore:

$$
\mathbf{t}_{factory}+R_y(\theta)\mathbf{p}_{desktop}
=
\begin{bmatrix}p_{xr,x}\\y_f\\p_{xr,z}\end{bmatrix}.
$$

In plain language, the virtual desktop start point is rotated to match the phone's heading and translated onto the user's real location. The selected surface supplies the floor height. This is why AR begins at the same internal factory location as desktop mode.


## 8.6 AR controls

After placement, exactly four machine buttons are shown:

<p class="table-caption"><b>Table 8.1:</b> Augmented-reality controls.</p>

| AR button | Result |
|---|---|
| Start Robot | Starts the local robot cycle and publishes the robot-start MQTT command |
| Stop Robot | Stops and resets the robot and publishes the stop command |
| Start Conveyor | Starts conveyor animation and publishes `room/StartButton` with payload `1` |
| Stop Conveyor | Stops conveyor animation and publishes `room/StopButton` with payload `1` |

Buttons are enabled or disabled from the current machine state so a running machine cannot be repeatedly started and an idle machine cannot be repeatedly stopped.

**Exit AR** is a separate top-level control fixed to the upper-right safe area. It calls `XRSession.end()` and remains available before and after factory placement.


## 8.7 Three-times movement and collision protection

WebXR owns the physical camera pose, so the application does not directly move the XR camera. Instead, it measures the horizontal camera displacement since the previous frame, transforms that movement into factory-local coordinates and requests three times the displacement inside the virtual factory.

For a physical movement $\Delta\mathbf{p}$:

$$
\Delta\mathbf{p}_{virtual}=3\,\Delta\mathbf{p}.
$$

The desired path is divided into substeps no larger than `0.06` scene units. Every substep is tested against the same collider collection used by desktop mode. X and Z are resolved separately to allow sliding. If a collision blocks the amplified path, the factory root is corrected so the user remains at the nearest safe virtual position.

This method provides faster exploration without allowing the amplified movement to skip through thin walls.


```mermaid
flowchart TD
    A["Measure physical camera displacement Δp"] --> B["Amplify horizontal travel\nΔp virtual = 3 × Δp"]
    B --> C["Divide the path into substeps\nMaximum length: 0.06 units"]
    C --> D{"Collision in the next substep?"}
    D -->|"No"| E["Resolve X and Z separately\nAllow safe wall sliding"]
    D -->|"Yes"| F["Reject the blocked component\nKeep the nearest safe position"]
    E --> G{"More substeps?"}
    F --> G
    G -->|"Yes"| D
    G -->|"No"| H["Update the factory root\nUser remains collision-safe"]

    classDef input fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px,color:#111827;
    classDef process fill:#fef3c7,stroke:#b45309,stroke-width:2px,color:#111827;
    classDef decision fill:#fce7f3,stroke:#be185d,stroke-width:2px,color:#111827;
    classDef safe fill:#dcfce7,stroke:#15803d,stroke-width:2px,color:#111827;
    classDef blocked fill:#fee2e2,stroke:#b91c1c,stroke-width:2px,color:#111827;
    class A input;
    class B,C process;
    class D,G decision;
    class E,H safe;
    class F blocked;
```

<p class="diagram-caption"><b>Figure 8.2:</b> Three-times AR movement with collision-safe substeps.</p>


## 8.8 AR-to-desktop transition

Ending the XR session performs a complete state restoration:

- cancel any active hit-test source;
- hide the reticle, AR control area and Exit button;
- clear held desktop keys and AR collision state;
- restore meshes hidden for AR;
- reset factory position, rotation and scale;
- restore desktop background, fog and shadows;
- restore the camera position `(8, 1.7, 25)`;
- restore yaw `−π/2`, pitch `0` and camera scale `1`;
- repeat the camera reset on the next animation frame so WebXR shutdown cannot overwrite it.

This prevents the distorted or misplaced desktop view that can occur when the final XR camera transform leaks into normal rendering.

<div class="warning"><b>Physical safety:</b> Virtual collision detection protects only against virtual walls and machines. It cannot detect every real object. Use AR in a clear indoor area away from stairs, roads, operating machinery, glass and other hazards. Keep attention on the real surroundings.</div>


# 9. MQTT and physical-system integration

## 9.1 Communication model

The browser uses MQTT.js over a secure WebSocket connection. The ESP8266 uses MQTT over TLS on port `8883`. HiveMQ Cloud is the central broker. Node-RED subscribes to the same topics and aggregates them into dashboard state.

Command payloads are normally `1`. State parsers also accept common representations such as `true`, `on`, `start`, `pressed`, `false`, `off`, `stop` and `released`, which makes the system tolerant of different MQTT publishers.

## 9.2 Topic map

<p class="table-caption"><b>Table 9.1:</b> MQTT topic map.</p>

| Topic | Typical publisher | Subscribers | Meaning / payload |
|---|---|---|---|
| `factory/room1/robot/start` | Browser control panel or AR button | ESP8266 | Start physical robot indication; payload `1` |
| `factory/room1/robot/stop` | Browser control panel or AR button | ESP8266 | Stop physical robot indication; payload `1` |
| `room/StartRobot` | ESP8266 | Browser, Node-RED | Robot start state/event; `1` active, `0` inactive |
| `room/StopRobot` | ESP8266 | Browser, Node-RED | Robot stop state/event; `1` active, `0` inactive |
| `room/StartButton` | Browser or physical conveyor start button | ESP8266, browser, Node-RED | Conveyor start command/event; payload `1` |
| `room/StopButton` | Browser or physical conveyor stop button | ESP8266, browser, Node-RED | Conveyor stop command/event; payload `1` |
| `room/MotorState` | ESP8266 | Browser, Node-RED | Conveyor motor state; `1` running, `0` stopped |
| `room/temperature` | ESP8266 | Browser, Node-RED | DHT11 temperature in degrees Celsius |
| `room/humidity` | ESP8266 | Browser, Node-RED | DHT11 relative humidity in percent |
| `room/LightIntensity` | ESP8266 | Browser, Node-RED | LDR value mapped and constrained to `0–100` |
| `factory/room1/conveyor/start` | Older simulation builds | Node-RED compatibility path | Converted to the same state as `room/StartButton` |
| `factory/room1/conveyor/stop` | Older simulation builds | Node-RED compatibility path | Converted to the same state as `room/StopButton` |

<div class="note"><b>Current Version 4 behavior:</b> the production browser publishes conveyor commands directly on `room/StartButton` and `room/StopButton`. The two `factory/room1/conveyor/...` topics remain in Node-RED only as backward-compatible inputs for older deployments.</div>


## 9.3 Event sequence examples

### 9.3.1 Browser starts the conveyor

1. The operator presses the 3D START button or **Start Conveyor** in AR.
2. The browser starts its conveyor animation immediately.
3. The browser publishes `room/StartButton = 1`.
4. The ESP8266 receives the command and sets `motorRunning = true`.
5. Motor outputs change to run the physical conveyor.
6. The ESP8266 publishes `room/MotorState = 1`.
7. Browser and Node-RED show the running state.

### 9.3.2 Physical button starts the robot

1. The start button on pin `D4` changes from HIGH to LOW.
2. The 50 ms debounce condition confirms a new press.
3. The yellow RobotOn LED turns on and the blue RobotOff LED turns off.
4. The ESP8266 publishes `room/StartRobot = 1` and `room/StopRobot = 0`.
5. The browser starts the robot animation.
6. Node-RED updates the robot actuator card.

## 9.4 Loop prevention

Browser actions pass a source value such as `"panel"` or `"ar"`. MQTT-received actions pass `"mqtt"`. Only panel/AR actions publish a new command, while actions triggered by an incoming MQTT state update the simulation without republishing the same command. This prevents an endless browser message loop.


# 10. ESP8266 hardware mapping

<p class="table-caption"><b>Table 10.1:</b> ESP8266/NodeMCU hardware mapping.</p>

| Device / signal | NodeMCU pin | Function |
|---|---|---|
| L298N IN1 | `D0` | Motor direction/output 1 |
| L298N IN2 | `D1` | Motor direction/output 2 |
| Conveyor START button | `D2` | Active-LOW input using `INPUT_PULLUP` |
| Conveyor STOP button | `D3` | Active-LOW input using `INPUT_PULLUP` |
| Robot START button | `D4` | Active-LOW input using `INPUT_PULLUP` |
| Robot STOP button | `D5` | Active-LOW input using `INPUT_PULLUP` |
| RobotOn LED | `D6` | Yellow running indicator |
| RobotOff LED | `D7` | Blue idle indicator |
| DHT11 data | `D8` | Temperature and humidity input |
| LDR | `A0` | Analog light input |

The L298N ENA input is expected to use its full-speed jumper; no NodeMCU enable pin is assigned. At startup, both motor outputs are forced LOW before Wi-Fi and MQTT setup, preventing unintended motor motion during network connection.

## 10.1 Timing behavior

<p class="table-caption"><b>Table 10.2:</b> Controller timing intervals.</p>

| Function | Interval |
|---|---:|
| Button debounce | 50 ms |
| LDR sampling | 500 ms |
| DHT11 and periodic state publication | 2,000 ms |
| MQTT reconnect attempt | 5,000 ms |

Most timing uses `millis()` instead of blocking delays, allowing buttons, MQTT and outputs to remain responsive. The Wi-Fi connection loop still waits during initial startup.

## 10.2 Motor logic

- Running: `D0 = HIGH`, `D1 = LOW`.
- Stopped: `D0 = LOW`, `D1 = LOW`.
- A physical START press publishes both `room/StartButton = 1` and `room/MotorState = 1`.
- A physical STOP press publishes both `room/StopButton = 1` and `room/MotorState = 0`.
- A remote command changes `motorRunning` and then publishes the resulting motor state.


# 11. Node-RED dashboards

Two flows provide complementary views:

- **Smart Factory dashboard (`flows.json`)** — combines environmental values, robot state, conveyor state and recent events.
- **Actuators and Sensors dashboard (`Actuators and Sensors.json`)** — emphasizes the four physical actuator-input signals together with sensor readings.

Each flow:

1. receives MQTT messages;
2. normalizes scalar, Boolean and numeric values;
3. stores the latest combined state in Node-RED flow context;
4. adds a source and timestamp;
5. sends one state object to the dashboard template.

Compatibility nodes translate:

- `factory/room1/conveyor/start` → `room/StartButton`;
- `factory/room1/conveyor/stop` → `room/StopButton`.

This ensures an older simulation command produces the same dashboard change as a physical conveyor button. The robot path is not translated because its command and state design already works correctly.


# 12. How to use the system

## 12.1 General prerequisites

- Use a current Chromium-based desktop browser for normal mode.
- Keep an internet connection available for Three.js, GLTFLoader, MQTT.js and HiveMQ.
- Serve the HTML from a web server instead of relying on direct `file://` access.
- Use HTTPS for AR.
- For physical MQTT integration, configure a private broker for the browser and ESP8266. The public repository copy runs without MQTT by default.
- Import and deploy the required Node-RED flow when dashboards are needed.

## 12.2 Start the desktop simulation

1. Place the production HTML in the web-server document directory.
2. Open its HTTP or HTTPS address in the desktop browser.
3. Wait until the factory and imported models are visible.
4. Click the scene to lock the mouse.
5. Use W/A/S/D or the arrow keys to move.
6. Aim the crosshair at a control-panel button and click.
7. Walk toward the internal door and verify it opens automatically.
8. Press **Show Meshes** when a wireframe demonstration is required.

## 12.3 Start AR

1. Open the HTTPS deployment in Chrome on an ARCore-compatible Android phone.
2. Confirm that the button reads **Start AR**, not **AR not supported** or **AR needs HTTPS**.
3. Press **Start AR** and grant camera permission.
4. Point the rear camera at a horizontal floor with good light and visible texture.
5. Move slowly until the green placement ring appears.
6. Stand facing the direction that should match the desktop starting view.
7. Tap the green ring once.
8. The phone is now mapped to the desktop factory start location.
9. Walk physically to navigate; virtual travel is amplified three times.
10. Use Start/Stop Robot and Start/Stop Conveyor at the bottom of the AR view.
11. Press **Exit AR** at the top-right to return to the desktop start pose.

## 12.4 Start the physical controller

1. Verify the motor driver, buttons, LEDs, DHT11 and LDR are connected to the documented pins.
2. Verify grounds are common and the NodeMCU receives a stable, suitable supply.
3. Upload the Arduino sketch with the required libraries installed.
4. Open the serial monitor at `115200` baud.
5. Confirm Wi-Fi connection and four successful MQTT subscriptions.
6. Test each physical button and observe the expected topic in HiveMQ or Node-RED.
7. Test a browser command and confirm the corresponding LED or motor response.

## 12.5 Start Node-RED monitoring

1. Import the chosen JSON flow.
2. Configure the MQTT broker node with the deployment's credentials.
3. Install the dashboard package required by the imported template.
4. Deploy the flow.
5. Open the dashboard and confirm sensor and actuator cards receive data.


# 13. Deployment and production requirements

## 13.1 Browser hosting

The HTML can be hosted by Nginx on a Raspberry Pi, Netlify or another static HTTPS service. AR requires a secure origin, with `localhost` being the normal development exception. Configure the server to return the HTML with UTF-8 content and allow access to the external CDN dependencies.

## 13.2 Performance characteristics

Production optimizations already present include:

- pixel ratio capped at `2`;
- XR framebuffer scale set to `0.85`;
- shadows disabled during AR;
- frame delta capped at `0.04 s`;
- repeated movement and collision vectors reused instead of recreated every frame;
- MQTT button-state updates skipped when no state changed;
- one animation loop for desktop and XR;
- embedded models and audio to reduce separate asset requests.

The 23.6 MB HTML remains a large initial download. For production at scale, separate hashed GLB and audio files with long cache headers can improve repeat load time, although doing so removes the single-file deployment advantage.

## 13.3 Security actions required before public release

<div class="warning"><b>Important:</b> The original browser HTML contained MQTT credentials. They have been removed from this repository copy. Rotate the original credentials before any further deployment.</div>

- Do not publish permanent MQTT credentials in browser JavaScript.
- Create a restricted browser account that can access only the required topics.
- Use a separate ESP8266 account with the minimum publish/subscribe permissions.
- Store Wi-Fi and firmware secrets outside the public repository.
- Replace `espClient.setInsecure()` with broker-certificate or CA validation.
- Disable anonymous broker access.
- Use unique MQTT client identifiers.
- Monitor failed connections and unexpected publishers.
- Apply a Content Security Policy and self-host dependencies if the deployment requires strict supply-chain control.


# 14. Testing and verification

## 14.1 Desktop acceptance tests

<p class="table-caption"><b>Table 14.1:</b> Desktop acceptance tests.</p>

| Test | Expected result |
|---|---|
| Initial load | Camera begins inside factory at `(8, 1.7, 25)` |
| W/S direction | W moves forward; S moves backward |
| Side movement | A/D and left/right arrows move laterally |
| Wall collision | Camera does not cross exterior or interior walls |
| Machine collision | Camera slides along equipment but cannot enter it |
| Door approach | Door opens before collision |
| Door departure | Door closes only when doorway is clear |
| Robot START | Full pick-and-place cycle and synchronized sound begin |
| Robot STOP | Robot resets and sound stops |
| Conveyor START | Rollers, belt, box and sound begin |
| Conveyor STOP | Movement and sound stop; box position is held |
| Wireframe | Show Meshes changes all meshes; Normal View restores materials |

## 14.2 AR acceptance tests

<p class="table-caption"><b>Table 14.2:</b> Augmented-reality acceptance tests.</p>

| Test | Expected result |
|---|---|
| Unsupported desktop | Desktop instructions remain; AR reports unsupported |
| HTTP deployment | AR reports that HTTPS is required |
| Compatible phone | Start AR opens camera after permission |
| Surface detection | Reticle appears only on a horizontal floor |
| Placement | Factory locks after one tap and hit testing stops |
| Start mapping | User begins at the desktop-equivalent internal position |
| Floor alignment | Factory floor and markings remain locked to selected surface |
| Movement scale | Measured virtual travel is approximately three times physical travel |
| AR collision | Amplified movement cannot skip through walls or machines |
| AR controls | Exactly four machine buttons appear after placement |
| Exit control | Exit AR remains visible at top-right |
| Exit reset | Desktop returns to original position and viewing direction |

## 14.3 MQTT and hardware acceptance tests

- Browser robot START/STOP produces the correct `factory/room1/robot/...` message.
- Browser conveyor START/STOP produces `room/StartButton` or `room/StopButton`.
- ESP8266 receives all four command subscriptions.
- Physical buttons publish only on new presses, not continuously.
- Motor state changes immediately after local or remote conveyor control.
- Node-RED actuator cards react identically to current and legacy conveyor topics.
- Temperature, humidity and light values update without blocking control input.
- Network reconnection does not start the motor unexpectedly.


# 15. Troubleshooting

<p class="table-caption"><b>Table 15.1:</b> Troubleshooting guide.</p>

| Symptom | Likely cause | Corrective action |
|---|---|---|
| AR button says **AR needs HTTPS** | Page is not secure | Open the HTTPS deployment |
| AR button says **AR not supported** | Browser/device lacks immersive AR | Use current Chrome on an ARCore-compatible Android phone and update Google Play Services for AR |
| Camera permission fails | Permission blocked | Allow camera access in browser/site settings and retry |
| No green reticle | Weak lighting, featureless floor or non-horizontal target | Improve lighting, point at a textured horizontal floor and move slowly |
| Factory cannot be placed | Reticle is not currently valid | Tap only while the green ring is visible |
| AR starts in wrong orientation | Phone was facing the wrong direction during placement | Exit AR, face the intended desktop direction and place again |
| AR movement passes an obstacle | Old build or placement/collision state did not initialize | Verify the production Version 4 build and re-enter AR |
| Exit returns to a distorted desktop view | Old session-end code | Use the build that performs immediate and next-frame desktop reset |
| Machine moves in browser but not hardware | ESP8266 offline, topic mismatch or broker failure | Check serial subscription status, broker connection and exact topic spelling |
| Physical button changes hardware but not dashboard | Node-RED not subscribed/deployed | Verify MQTT nodes, broker configuration and flow deployment |
| Conveyor dashboard differs between old and current simulations | Legacy topic not normalized | Confirm compatibility change nodes are present |
| DHT11 values are invalid | Wiring, power or incorrect D8 connection | Verify DHT11 data on D8 and common ground |
| Motor starts during boot | Incorrect wiring or old firmware | Use the documented build that drives both motor outputs LOW before Wi-Fi setup |
| HTML loads without models | CDN unavailable or unsupported direct-file execution | Use a local/hosted web server and internet access |


# 16. Maintenance reference

## 16.1 Important production constants

<p class="table-caption"><b>Table 16.1:</b> Important production constants.</p>

| Constant / value | Purpose |
|---|---|
| `DESKTOP_START_POSITION = (8, 1.7, 25)` | Shared desktop and AR logical start |
| `DESKTOP_START_YAW = −π/2` | Initial desktop viewing direction |
| `DESKTOP_START_PITCH = 0` | Initial vertical view |
| `AR_MOVEMENT_SCALE = 3` | Virtual-to-physical horizontal travel ratio |
| `AR_COLLISION_STEP = 0.06` | Maximum collision-probe substep |
| collision half-width `0.32` | User collision body in X and Z |
| collision height `0.05–1.85` | User collision body vertical range |
| normal speed `3.2` | Desktop walking speed |
| Shift speed `5.2` | Desktop fast movement |
| frame-delta cap `0.04` | Maximum simulation time step |
| XR framebuffer scale `0.85` | Mobile XR performance balance |
| robot cycle `7.340408 s` | Animation/audio synchronization |


# 17. References and official documentation

The following primary and official documentation supports the technologies described in this notebook:

1. Three.js, **Three.js Documentation**: https://threejs.org/docs/
2. Three.js, **BoxGeometry**: https://threejs.org/docs/pages/BoxGeometry.html
3. Three.js, **GLTFLoader**: https://threejs.org/docs/pages/GLTFLoader.html
4. Three.js, **WebXRManager**: https://threejs.org/docs/pages/WebXRManager.html
5. W3C, **WebXR Device API**: https://www.w3.org/TR/webxr/
6. W3C, **WebXR Hit Test Module**: https://www.w3.org/TR/webxr-hit-test-1/
7. W3C, **WebXR DOM Overlays Module**: https://www.w3.org/TR/webxr-dom-overlays-1/
8. OASIS Open, **MQTT Version 5.0**: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
9. HiveMQ, **HiveMQ Cloud Quick Start Guide**: https://docs.hivemq.com/hivemq-cloud/quick-start-guide.html
10. HiveMQ, **MQTT Listener Documentation**: https://docs.hivemq.com/hivemq/latest/user-guide/listeners.html
11. Node-RED, **Documentation**: https://nodered.org/docs/
12. Node-RED, **Core Concepts**: https://nodered.org/docs/user-guide/concepts
13. ESP8266 Arduino Core, **ESP8266WiFi Reference**: https://arduino-esp8266.readthedocs.io/en/latest/esp8266wifi/readme.html
14. ESP8266 Arduino Core, **Digital I/O Reference**: https://arduino-esp8266.readthedocs.io/en/latest/reference.html
15. Nick O'Leary, **PubSubClient MQTT Library**: https://github.com/knolleary/pubsubclient

---


# Appendix A. AI-supported review prompts

This appendix records examples of limited AI assistance used during the project. The prompts focus on technically demanding review, troubleshooting and explanation tasks. They do not present AI as the designer or implementer of the Smart Factory. Suggestions produced from these prompts must be checked against the project source files, measurements and practical tests before being accepted.

<p class="table-caption"><b>Table A.1:</b> AI-supported review prompts and purposes.</p>

| ID | Complex topic | Main purpose |
|---|---|---|
| A.1 | MQTT command/state consistency | Review topic direction and prevent feedback loops |
| A.2 | WebXR plane placement | Check stable floor anchoring and camera mapping |
| A.3 | Three-times AR navigation | Review amplified movement with collision safety |
| A.4 | Desktop-to-AR state transition | Diagnose pose and state conflicts between modes |
| A.5 | ESP8266 event handling | Review debounce, non-blocking timing and publishing |
| A.6 | Node-RED state normalization | Align simulation and physical dashboard behavior |
| A.7 | Production optimization | Identify safe cleanup without changing behavior |
| A.8 | Technical documentation | Organize and quality-check the final notebook |

## A.1 MQTT command and state-topic consistency

**Prompt**

> Review the supplied browser simulation, ESP8266 sketch and Node-RED flow topic maps. Identify mismatches between command topics, physical-input topics and resulting-state topics. Recommend a consistent publish/subscribe mapping that preserves the existing robot behavior, makes simulated and physical conveyor controls produce the same dashboard result, and prevents MQTT feedback loops. Explain the direction of every recommended topic and do not change unrelated machine logic.

**Purpose:** Used to compare the three communication layers and support correction of conveyor-topic inconsistencies without redesigning the robot subsystem.

## A.2 Stable WebXR floor placement and camera mapping

**Prompt**

> Analyze the WebXR placement logic in the provided HTML. The operator must select one horizontal real surface, after which the factory floor, safety lines and hazard markings must remain locked to that surface. Determine how the selected hit-test pose, XR camera heading and desktop start position `(8, 1.7, 25)` should be combined so that AR begins at the same logical factory location as desktop mode. Highlight causes of drift, repeated scanning or incorrect fixed offsets.

**Purpose:** Used to review the placement mathematics and identify why earlier AR versions began outside the intended internal factory position.

## A.3 Three-times AR movement with collision protection

**Prompt**

> Review an AR navigation method in which one metre of physical horizontal movement should produce three metres of virtual movement. Propose a collision-safe algorithm that transforms XR camera displacement into factory-local coordinates, divides the amplified path into small substeps, resolves X and Z independently, and prevents movement through walls, machines or furniture. Keep door openings usable and explain how tunnelling through thin colliders is avoided.

**Purpose:** Used to examine the relationship between movement amplification, coordinate conversion and the existing desktop collision model.

## A.4 Reliable transition between desktop and AR modes

**Prompt**

> Inspect the supplied WebXR session lifecycle for conflicting desktop and AR states. Identify which keyboard states, pointer-lock state, camera transforms, factory-root transforms, hidden meshes, fog, shadows, hit-test objects and collision history must be reset when entering or leaving AR. The final result must restore the exact desktop start pose after `XRSession.end()` and must not allow the browser's final XR frame to overwrite that reset.

**Purpose:** Used to troubleshoot the distorted view and collision conflict that occurred when switching between normal and AR modes.


## A.5 ESP8266 event-driven control and non-blocking timing

**Prompt**

> Review the provided ESP8266 control sketch for the conveyor motor, robot indicators, push-buttons, DHT11 and LDR. Check active-LOW button handling, 50 ms debounce, `millis()`-based scheduling, initial motor-off behavior, MQTT reconnection and event-only publishing. Identify any logic that could delay STOP, publish repeatedly without a new input, or start the motor during boot. Keep the documented pin assignments and intended machine behavior unchanged.

**Purpose:** Used to support a focused reliability review of the physical controller while preserving the existing wiring and control concept.

## A.6 Node-RED normalization of simulation and physical inputs

**Prompt**

> Compare the supplied Node-RED dashboard flows with the MQTT messages produced by the browser and ESP8266. Design a normalization step that converts current and legacy conveyor commands into the same `StartButton` and `StopButton` state representation used by physical inputs. Preserve source and timestamp information, avoid altering the robot path, and explain how one combined state object should update the actuator cards.

**Purpose:** Used to diagnose why a browser command changed the conveyor but did not update the actuator-input cards in the same way as a physical button press.

## A.7 Production optimization without functional regression

**Prompt**

> Perform a static review of the provided single-file Smart Factory HTML for production readiness. Identify duplicated handlers, unused variables, repeated graphics allocations, per-frame object creation and unreachable UI rules that can be removed safely. Preserve the approved Three.js scene, MQTT topics, audio and model assets, desktop navigation, WebXR placement, three-times AR movement, collision protection, four AR machine controls and Exit AR reset. Separate safe cleanup from changes that would alter behavior.

**Purpose:** Used to guide a conservative cleanup review after functional testing, with emphasis on avoiding regression in accepted features.

## A.8 Organization and quality review of the documentation

**Prompt**

> Using only the supplied Smart Factory HTML, ESP8266 sketch, Node-RED flows and verified operating values, help organize a comprehensive Jupyter notebook. Structure the material into consistently numbered sections covering architecture, the 3D factory, desktop operation, WebXR AR, MQTT, hardware, Node-RED, operating instructions, deployment, testing and troubleshooting. Suggest where Mermaid diagrams, tables, equations and short readable code excerpts improve clarity. Do not claim authorship of the system, invent test results, expose credentials or imply that AI designed or implemented the project.

**Purpose:** Used to help organize, phrase and quality-check the documentation after the system implementation and testing work had been completed.


# Appendix B. HTML source-code organization and explanation

The visual simulation is contained in one HTML file. MQTT connection settings are loaded from an optional, untracked config.local.js. Its JavaScript has been compacted and its GLB and MP3 assets are stored as data URIs, so the raw file is not arranged like a multi-file development project. This appendix divides the implementation into logical maintenance sections. The short excerpts use descriptive names to express the implemented logic clearly; they are explanatory representations rather than a second executable version of the production file.

<p class="table-caption"><b>Table B.1:</b> Functional organization of the production HTML source.</p>

| Section | Main HTML/JavaScript elements | Responsibility |
|---|---|---|
| B.1 | `<head>`, CSS, HUD and AR overlay elements | Page layout, instructions and responsive controls |
| B.2 | Import map, Three.js, `GLTFLoader`, MQTT.js and data URIs | Dependencies and embedded assets |
| B.3 | Scene, camera, renderer and factory placement root | Core Three.js hierarchy and rendering configuration |
| B.4 | Geometry helpers, materials, textures, lights and GLB loading | Construction and appearance of the factory |
| B.5 | Robot, conveyor, door, sound and status logic | Machine state and animation behavior |
| B.6 | Pointer lock, keyboard state and raycasting | Desktop navigation and control-panel interaction |
| B.7 | Static/dynamic colliders and safe-axis movement | Desktop and AR obstacle protection |
| B.8 | MQTT connection, subscriptions, parsing and publishing | Digital/physical synchronization |
| B.9 | WebXR support gate, session and hit testing | AR compatibility and surface detection |
| B.10 | Placement, 3× movement and Exit AR cleanup | Stable AR operation and return to desktop |
| B.11 | Animation loop, resizing and state-change updates | Coordinated execution and performance |

## B.1 Document structure, styling and interface overlays

The `<head>` defines UTF-8 text, the responsive viewport and the Smart Factory page title. CSS then separates desktop and AR controls. Important elements are `#hud`, `#cross`, `#meshViewButton`, `#startArButton`, `#arStartupStatus`, `#arHud`, `#arMachineControls` and the independent `#exitArButton`.

```html
<button id="startArButton" type="button">Start AR</button>
<div id="arHud" hidden>...</div>
<button id="exitArButton" type="button" hidden>Exit AR</button>
```

The `hidden` attribute and the `body.ar-session` class control which interface is visible. Keeping **Exit AR** outside the lower AR panel prevents it from being clipped on small screens. Desktop instructions remain unchanged until WebXR compatibility is positively confirmed.

## B.2 Libraries and embedded assets

The import map resolves the `three` module. `GLTFLoader` is imported as an ES module, while MQTT.js is loaded as a browser script. Robot audio, conveyor audio and GLB models are embedded as base64 data URIs. This produces a large but portable HTML file.

```html
<script type="importmap">
  {"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js"}}
</script>
<script src="https://unpkg.com/mqtt/dist/mqtt.min.js"></script>
```

The external libraries still require network access unless they are hosted locally. Embedded binary data must not be edited as ordinary text because a single changed character can corrupt a model or sound.

## B.3 Scene, camera, renderer and placement hierarchy

The core Three.js objects are the scene, perspective camera and WebGL renderer. The desktop camera starts at `(8, 1.7, 25)`. A placement root sits above the complete factory group in the scene graph. Desktop mode keeps that root at the origin; AR mode translates and rotates it to map the desktop start point onto the selected real surface.

```javascript
const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(75, aspect, 0.05, 140);
camera.position.set(8, 1.7, 25);
const factoryPlacementRoot = new THREE.Group();
factoryPlacementRoot.add(factoryGroup);
scene.add(factoryPlacementRoot);
```

The renderer enables antialiasing, transparency, soft shadows and WebXR. Pixel ratio and XR framebuffer scale are capped to reduce mobile GPU load.

## B.4 Factory construction and visual assets

Procedural helper functions create repeated boxes, cylinders, panels, rollers, rails and text surfaces. These are assembled into the floors, walls, roof, lighting, control cabinets, safety fences, conveyor, desks, racks and other equipment. Physically based materials distinguish painted metal, concrete, glass and plastic. Canvas-generated textures provide concrete variation, floor markings and hazard patterns. `GLTFLoader` adds the embedded robot, workstation and chair models.

The important design separation is:

1. geometry determines shape;
2. material and texture determine appearance;
3. transforms determine placement;
4. the scene graph groups parts that move together;
5. collider registration defines which parts block navigation.


## B.5 Machine, door, audio and status logic

The robot and conveyor are controlled by explicit running states. Start/stop functions update the local state immediately, change the control-panel indicators, control audio and publish a command only when the action originated from the desktop panel or AR interface. MQTT-originated updates use the same local functions without publishing again.

```javascript
function startConveyor(source = "panel") {
  conveyorRunning = true;
  conveyorSound.play();
  updateConveyorButtons();
  if (source !== "mqtt") publish("room/StartButton", "1");
}
```

The robot uses the cycle time to calculate joint and claw poses. The conveyor uses elapsed time to rotate rollers, scroll the belt texture and move the package. The automatic door uses distance thresholds plus a doorway-clear check before closing. Machine states are updated in the shared animation loop rather than by several competing timers.

## B.6 Desktop input and control-panel selection

Keyboard listeners store whether movement keys are currently held. Mouse movement changes yaw and pitch only while pointer lock is active. Each animation frame converts the stored input into a time-scaled movement vector, so movement speed remains consistent across different frame rates.

```javascript
const speed = shiftHeld ? 5.2 : 3.2;
const distance = speed * Math.min(frameDelta, 0.04);
```

A raycaster projects from the center crosshair into the 3D scene. When the user clicks, the nearest registered control-panel target determines whether the robot or conveyor start/stop action is executed.

## B.7 Collision organization

The collision section maintains both static and dynamic axis-aligned boxes. Static entries cover fixed walls, machines, furniture and safety structures. Dynamic boxes are refreshed for moving elements such as the automatic door. The camera is represented by a small user collision body.

Desktop movement tests the X component and then the Z component. This permits sliding along an obstacle. AR movement uses the same collision collection, but it divides the amplified route into substeps no longer than `0.06` units.

```javascript
function moveSafely(deltaX, deltaZ) {
  tryAxis("x", deltaX);
  tryAxis("z", deltaZ);
}
```

The separation between collider creation, collider refresh and movement resolution is important. Geometry that is visible but not registered does not block the user; a collider that is not refreshed may remain at an old moving-object position.

## B.8 MQTT communication and feedback-loop prevention

The MQTT block opens a secure WebSocket connection to HiveMQ, subscribes to machine-state and sensor topics, parses incoming payloads and routes each topic to the correct local update. Browser commands use the deployed topic mapping:

```javascript
publish("factory/room1/robot/start", "1");
publish("factory/room1/robot/stop", "1");
publish("room/StartButton", "1");
publish("room/StopButton", "1");
```

Incoming state topics update animation, indicators and sensor displays. The source parameter distinguishes a local operator action from a broker-originated update. Because only local panel/AR actions publish, an incoming MQTT message cannot generate an endless republishing loop.


## B.9 WebXR support, session start and surface hit testing

`checkArSupport()` first verifies HTTPS, `navigator.xr` and support for `immersive-ar`. The **Start AR** button is enabled only after this check succeeds. `startArSession()` requests hit testing and optionally requests DOM overlay and local-floor support.

```javascript
const session = await navigator.xr.requestSession("immersive-ar", {
  requiredFeatures: ["hit-test"],
  optionalFeatures: ["dom-overlay", "local-floor"],
  domOverlay: { root: document.body }
});
await renderer.xr.setSession(session);
```

During placement, a viewer-space ray is tested against the real environment. A ring reticle is shown only when a sufficiently horizontal surface is detected. One selection event records the floor height and camera heading, places the factory, cancels the hit-test source and hides the reticle. Cancelling hit testing is what prevents later scanning from changing the selected factory floor.

## B.10 AR placement, movement amplification and session exit

Placement rotates the factory root to the current horizontal phone heading and translates it so the logical desktop start position coincides with the XR camera's real location. After placement, physical XR camera displacement is converted into factory-local movement and multiplied by `3`. `startArCollision()`, `moveArAxisSafely()` and `updateArCollision()` maintain the collision-safe virtual position.

The Exit button calls `XRSession.end()`. Session cleanup cancels hit testing, hides AR-only controls, restores meshes and rendering settings, clears desktop and AR input state, resets the factory root, and calls `resetDesktopCamera()`. The reset is repeated on the next browser frame because the final WebXR frame can otherwise overwrite the restored camera pose.

```javascript
exitArButton.addEventListener("click", () => currentSession?.end());

function resetDesktopCamera() {
  camera.position.set(8, 1.7, 25);
  yaw = -Math.PI / 2;
  pitch = 0;
  camera.scale.setScalar(1);
}
```

## B.11 Animation loop, resizing and production behavior

One renderer animation loop coordinates navigation, machine animation, door logic, collision refresh, AR processing, audio and rendering. State-dependent work is skipped when a machine is idle or when the interface has not changed. Reusable vectors reduce temporary allocations inside high-frequency movement and collision code.

```javascript
renderer.setAnimationLoop((time, xrFrame) => {
  updateNavigation(time);
  updateMachines(time);
  updateDoor();
  if (xrFrame) updateAugmentedReality(xrFrame);
  renderer.render(scene, camera);
});
```

The resize handler updates the camera aspect ratio, projection matrix and renderer size. Centralizing these updates and using one animation loop prevents duplicate desktop/AR loops from competing during a mode transition.

## B.12 Recommended maintenance order

When the production HTML must be changed, use the following order:

1. back up the tested production file;
2. identify the logical section in Table B.1;
3. avoid editing embedded base64 model and audio strings;
4. make one focused behavior change;
5. check HTML and module syntax;
6. test desktop navigation, machines and MQTT;
7. test AR placement, 3× collision handling and Exit AR on a compatible phone;
8. confirm that ending AR restores the desktop start pose;
9. record the verified change in this notebook.

This organization makes the single-file implementation maintainable while preserving its portable deployment format.
